# Reed-Solomon Decoder

In [1]:
%load_ext autoreload
%autoreload 1
%aimport classes.galois_field
%aimport classes.galois_poly

from classes.galois_field import *
from classes.galois_poly import *

In [81]:
gf_extended = GaloisField(4, 0b10011)
t           = 3
root        = [GFE1, 1, 2, 3, 4, 5]

def do_unpack(value, degree, gf):
    # useful return of a x**degree poly
    if value == -1:
        ak  = [GFE0]*degree + [GFE1]
    else:
        mask    = gf._order
        ak      = [(value >> i*gf._field_m) & mask for i in range(degree+1)]
        ak      = [gf._get_power(a) for a in ak]
    return GaloisPoly(gf, ak[::-1])

def get_syndromes(r):
    return [r(a) for a in root]

def get_discrepancy(mu, syn, Lx):
    d_mu    = syn[mu]
    for i in range(1, Lx._degree+1):
        term    = gf_extended.mul(Lx @ i, syn[mu - i])[ID_POWER]
        d_mu    = gf_extended.add(d_mu, term)[ID_POWER]
    return d_mu

def get_locator_poly(syn):
    Lx      = []
    dmu     = []

    # step0 initialization: locator == 1, rho == -1
    Lx.append(do_unpack(-1, 0, gf_extended))
    rho = -1
    for mu in range(0, 2*t):
        disc = get_discrepancy(mu, syn, Lx[-1])

        # if dicrepancy zero (maybe S0 == 0), locator nor parameters change
        if disc == GFE0:
            poly = Lx[-1]
        else:
            if rho == -1:
                d_rho   = GFE1
                L_rho   = do_unpack(-1, 0, gf_extended)
            else:
                d_rho   = dmu[rho]
                L_rho   = Lx[rho]

            xm      = do_unpack(-1, mu-rho, gf_extended)
            xm      = xm * gf_extended.div(disc, d_rho)[ID_POWER]
            poly    = Lx[-1] + (xm * L_rho)
            # rho changes when the degree increases, this garantices non-zero discrepancy
            if poly._degree > Lx[-1]._degree:
                rho = mu

        dmu.append(disc)
        Lx.append(poly)
        # print(dmu, rho)
        # print(poly)
        # print()

    return Lx[-1]

Field Closed Succesfully!, 15 Non-Zero Elements


## Get Syndromes

In [82]:
# r   = 0x52D0A_F17CB_1EE64
r   = 0x56D09_F171B_1E06F
r   = do_unpack(r, 14, gf_extended)

syn = get_syndromes(r)

syn


[1, 14, 0, 10, 9, 13]

## Berlekamp-Massey Algorithm

In [83]:
Lx  = get_locator_poly(syn)


## Chien Algorithm (wip)

In [84]:
def _chien_search(Lx, gf):
    err_pos = []
    for i in range(gf._order):
        a_inv = gf.inv(i)[ID_POWER]         # alpha^-i (power index)
        if Lx(a_inv) == GFE0:               # root of the locator polynomial
            err_pos.append(i)
    if len(err_pos) != Lx._degree:
        print(f"Warning: {len(err_pos)} roots found, expected {Lx._degree} (uncorrectable)")
    return err_pos

# loc_root    = [13, 5]
# loc_root    = [GFE1, 2, 9]
loc_root    = [gf_extended.inv(p)[ID_POWER] for p in _chien_search(Lx, gf_extended)]
loc_root

[0, 9, 2]

## Forney Algorithm

In [85]:
def get_error_mask(gf, syn, Lx, loc_root, lo=0):
    # get evaluator poly, and locator derivative
    Sx  = GaloisPoly(gf, syn[::-1])
    Zx  = Sx * Lx
    Zx  = Zx % do_unpack(-1, Lx._degree, gf)
    Lxp = Lx.formal_derivative()
    print(Sx, Zx, Lxp)

    ek = []
    for rk in loc_root:
        num = gf.mul(Zx(rk), gf.pow(rk, lo-1)[ID_POWER])[ID_POWER]
        den = Lxp(rk)
        ek.append(gf.div(num, den)[ID_POWER])
    return ek

In [86]:
# correction
ek  = get_error_mask(gf_extended, syn, Lx, loc_root)
print(ek)

ex  = GaloisPoly(gf_extended, [GFE0])
for e, rk in zip(ek, loc_root):
    # get error positions from loc roots
    k   = gf_extended.inv(rk)[ID_POWER]
    if k == GFE0:
        k = 0
    elif k == GFE1:
        k = 1
    # accumulate terms of error poly
    xk  = do_unpack(-1, k, gf_extended)
    ex  = ex + (xk * e)

print(ex)
print(r + ex)

a13*x5 + a9*x4 + a10*x3 + x2 + a14*x + a1
 a8*x2 + a14*x + a1
 a4*x2
[7, 13, 2]
a2*x13 + a13*x6 + a7*x
a8*x14 + a1*x13 + a13*x12 + a14*x10 + a12*x9 + x8 + a10*x7 + a6*x6 + a7*x5 + x4 + a11*x3 + a13*x + a12



## Annex B Equations Compared To Forney

In [117]:
# using shorten-code, zero message sent, only error patterns present
r   = 0x00D00_F0000_0E000
r   = do_unpack(r, 14, gf_extended)

print("Implicit Error Masks:")
print(r)

# get the locator poly
syn = get_syndromes(r)
Lx  = get_locator_poly(syn)

# get roots of reciprocal, error position
Lx          = Lx.get_reciprocal()
loc_root    = [gf_extended.inv(p)[ID_POWER] for p in _chien_search(Lx, gf_extended)]
loc_root

# forney to get the error mask
ek_forney   = get_error_mask(gf_extended, syn, Lx, loc_root)

# # solve the system to get the error mask
# # Si  == ek*(pk)^i
# # 
# # S0  == e0*1       + e1*1      + e2*1      => | S0 |   == | 1      , 1     , 1     || e0   |
# # S1  == e0*p0      + e1*p1     + e2*p2     => | S1 |   == | p0     , p1    , p2    || e1   |
# # S2  == e0*p0^2    + e1*p1^2   + e2*p2^2   => | S2 |   == | p0^2   , p1^2  , p2^2  || e2   |
# r1  = GaloisPoly(gf_extended, [GFE1                                     , GFE1                                      , GFE1                                      , syn[0]])
# r2  = GaloisPoly(gf_extended, [loc_root[0]                              , loc_root[1]                               , loc_root[2]                               , syn[1]])
# r3  = GaloisPoly(gf_extended, [gf_extended.pow(loc_root[0],2)[ID_POWER] , gf_extended.pow(loc_root[1],2)[ID_POWER]  , gf_extended.pow(loc_root[2],2)[ID_POWER]  , syn[2]])

# r2  = r2 + (r1 * (r2 @ 3))
# r3  = r3 + (r1 * (r3 @ 3))

# r1  = r2 + (r1 * (r2 @ 2))
# r3  = (r3 * gf_extended.inv(r3 @ 2)[ID_POWER]) + (r2 * gf_extended.inv(r2 @ 2)[ID_POWER])

# r1  = (r1 * gf_extended.inv(r1 @ 1)[ID_POWER]) + (r3 * gf_extended.inv(r3 @ 1)[ID_POWER])
# r2  = r2 + r3

# r1  = r1 * (gf_extended.inv(r1 @ 3)[ID_POWER])
# r2  = r2 * (gf_extended.inv(r2 @ 2)[ID_POWER])
# r3  = r3 * (gf_extended.inv(r3 @ 1)[ID_POWER])

# print(r1)
# print(r2)
# print(r3)

# ek_system   = [r1 @ 0, r2 @ 0, r3 @ 0]
# syn, loc_root, ek_system, ek_forney, gf_extended(11)
# # the complement of masks must be identical to the one by forney, and dentical to coeffs of r

print(r)
ek_forney


Implicit Error Masks:
a13*x12 + a12*x9 + a11*x3
a6*x5 + a12*x4 + a12*x3 + a11*x2 + a1*x + a6
 x2 + a6*x + 1 x2 + a1

a13*x12 + a12*x9 + a11*x3


[4, 0, 11]